In [1]:
!pip install torchsummary

In [2]:
import os
import numpy as np
import cv2
import pickle
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchsummary import summary

In [139]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [140]:
# untar
!ls
!tar -xvzf dataset.tar.gz
# load train
train_images = pickle.load(open('train_images.pkl', 'rb'))
train_labels = pickle.load(open('train_labels.pkl', 'rb'))
# load val
val_images = pickle.load(open('val_images.pkl', 'rb'))
val_labels = pickle.load(open('val_labels.pkl', 'rb'))

'ls' is not recognized as an internal or external command,
operable program or batch file.
x train_images.pkl
x train_labels.pkl
x val_images.pkl
x val_labels.pkl


In [141]:
train_images = torch.tensor(train_images, dtype=torch.float32)
val_images = torch.tensor(val_images, dtype=torch.float32)

train_images = train_images.permute(0, 3, 1, 2)
val_images = val_images.permute(0, 3, 1, 2)

train_dataset = TensorDataset(train_images,
                              torch.tensor(train_labels.squeeze(), dtype=torch.long))
val_dataset = TensorDataset(val_images,
                            torch.tensor(val_labels.squeeze(), dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)

In [142]:
class ConvNet(nn.Module):
    def __init__(self):
        super(ConvNet, self).__init__()

        self.model = nn.Sequential(
            # First block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(3, 32, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Second block: Conv -> ReLU -> Conv -> ReLU -> MaxPool -> Dropout
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=True),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=0, bias=True),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Dropout(0.25),

            # Flatten layer
            nn.Flatten(),

            # Fully connected block: Dense -> ReLU -> Dropout -> Dense -> Softmax
            nn.Linear(1024, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 5),
        )

    def forward(self, x):
        return self.model(x)

In [143]:
model = ConvNet()

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0001, weight_decay=1e-6)

In [144]:
def train_one_epoch(model, train_loader, optimizer, criterion, device):
    model.train()  # Set model to training mode
    running_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the training loop
    train_loader_tqdm = tqdm(train_loader, desc="Training", leave=False)

    for inputs, labels in train_loader_tqdm:
        optimizer.zero_grad()  # Zero the parameter gradients
        inputs = inputs.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(inputs)
        loss = criterion(outputs, labels)

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Track loss and accuracy
        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

        # Update tqdm description with current loss and accuracy
        train_loader_tqdm.set_postfix(loss=running_loss / total, accuracy=100 * correct / total)

    train_accuracy = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_accuracy

In [145]:
def validate(model, val_loader, criterion, device):
    model.eval()  # Set model to evaluation mode
    val_loss = 0.0
    correct = 0
    total = 0

    # Progress bar for the validation loop
    val_loader_tqdm = tqdm(val_loader, desc="Validation", leave=False)

    with torch.no_grad():  # Disable gradient calculations for validation
        for inputs, labels in val_loader_tqdm:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            # Track loss and accuracy
            val_loss += loss.item()
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)

            # Update tqdm description with current validation loss and accuracy
            val_loader_tqdm.set_postfix(loss=val_loss / total, accuracy=100 * correct / total)

    val_accuracy = 100 * correct / total
    val_loss = val_loss / len(val_loader)
    return val_loss, val_accuracy

## **Start Activation Pruning**

In [146]:
# reload full model
model = ConvNet().to(device)
model.load_state_dict(torch.load('original_model.pt'))
print("Loaded original model weights.")


Loaded original model weights.


C:\Users\nickc\AppData\Local\Temp\ipykernel_21168\3605758991.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load('original_model.pt'))


In [150]:
activation_storage = {}

def get_activation_hook(name):
    def hook(model, input, output):
        if output.dim() == 4:  # Conv2d output: [batch, channels, H, W]
            stats = output.detach().mean(dim=(2, 3))  # [batch, channels]
        elif output.dim() == 2:  # Linear output: [batch, features]
            stats = output.detach()  # [batch, features]
        else:
            raise ValueError(f"Unexpected output shape {output.shape} for layer {name}")

        if name not in activation_storage:
            activation_storage[name] = []
        activation_storage[name].append(stats.cpu())
    return hook

In [151]:
hooks = []
for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        hooks.append(module.register_forward_hook(get_activation_hook(name)))


In [152]:
model.eval()
with torch.no_grad():
    for i, (inputs, _) in enumerate(train_loader):
        if i >= 200:  # subset of training data
            break
        inputs = inputs.to(device)
        _ = model(inputs)

IndexError: Dimension out of range (expected to be in range of [-2, 1], but got 2)

In [127]:
avg_activations = {}
for layer_name, activations in activation_storage.items():
    all_acts = torch.cat(activations, dim=0)  # [num_samples, channels]
    avg_per_filter = all_acts.mean(dim=0)     # [channels]
    avg_activations[layer_name] = avg_per_filter

# recommended to clean up hooks
for h in hooks:
    h.remove()

In [ ]:
k_percent = 0.20  # prune bottom 20%
pruned_indices = {}

for name, module in model.named_modules():
    if name in avg_activations:
        avg = avg_activations[name]
        num_units = avg.shape[0]
        num_prune = int(k_percent * num_units)
        prune_idx = torch.argsort(avg)[:num_prune]
        pruned_indices[name] = prune_idx

        with torch.no_grad():
            if isinstance(module, nn.Conv2d):
                module.weight[prune_idx] = 0
                if module.bias is not None:
                    module.bias[prune_idx] = 0
            elif isinstance(module, nn.Linear):
                module.weight[prune_idx, :] = 0
                if module.bias is not None:
                    module.bias[prune_idx] = 0

print("Pruning complete")

Pruning complete


In [ ]:
weight_masks = {}

for name, module in model.named_modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        mask = torch.ones_like(module.weight)

        if name in pruned_indices:
            idx = pruned_indices[name]
            
            # Conv2d: zero out filters (along output channels)
            if isinstance(module, nn.Conv2d):
                mask[idx] = 0
            
            # Linear: zero out neurons (rows of the weight matrix)
            elif isinstance(module, nn.Linear):
                mask[idx, :] = 0

        weight_masks[name] = mask


In [ ]:
def train_one_epoch_with_mask(model, train_loader, optimizer, criterion, device, weight_masks):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for inputs, labels in tqdm(train_loader, desc="Fine-Tuning", leave=False):
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()

        with torch.no_grad():
            for name, module in model.named_modules():
                if name in weight_masks:
                    if module.weight.grad is not None:
                        module.weight.grad *= weight_masks[name]

                    if module.bias is not None and module.bias.grad is not None:
                        if isinstance(module, nn.Conv2d):
                            # Conv2d: bias mask matches output channels
                            module.bias.grad *= weight_masks[name][:, 0, 0, 0]
                        elif isinstance(module, nn.Linear):
                            # Linear: bias mask matches output neurons
                            module.bias.grad *= weight_masks[name][:, 0]

        optimizer.step()

        running_loss += loss.item()
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)

    train_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    return train_loss, train_acc


In [131]:
val_loss, val_accuracy = validate(model, val_loader, criterion, device)
print(f"Post-Pruning Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")


Post-Pruning Val Loss: 1.3762, Val Accuracy: 41.23%


In [132]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5, weight_decay=1e-6)

# fine-tuning epochs
for epoch in range(8):
    train_loss, train_accuracy = train_one_epoch_with_mask(model, train_loader, optimizer, criterion, device, weight_masks)
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)
    print(f'[Fine-Tuning Epoch {epoch+1}] Val Accuracy: {val_accuracy:.2f}%')



[Fine-Tuning Epoch 1] Val Accuracy: 54.77%


[Fine-Tuning Epoch 2] Val Accuracy: 55.96%


[Fine-Tuning Epoch 3] Val Accuracy: 56.59%


[Fine-Tuning Epoch 4] Val Accuracy: 57.07%


[Fine-Tuning Epoch 5] Val Accuracy: 57.82%


[Fine-Tuning Epoch 6] Val Accuracy: 57.94%


[Fine-Tuning Epoch 7] Val Accuracy: 58.65%


[Fine-Tuning Epoch 8] Val Accuracy: 58.77%


In [133]:
val_loss, val_accuracy = validate(model, val_loader, criterion, device)
print(f"Post-Pruning AND post-fine tuning Val Loss: {val_loss:.4f}, Val Accuracy: {val_accuracy:.2f}%")

Post-Pruning AND post-fine tuning Val Loss: 1.0575, Val Accuracy: 58.77%


In [134]:
def count_zeroed_and_total_weights(model):
    total_params = 0
    zero_params = 0

    for name, param in model.named_parameters():
        if "weight" in name:
            total_params += param.numel()
            zero_params += (param == 0).sum().item()

    print(f"Total weights     : {total_params:,}")
    print(f"Zeroed weights    : {zero_params:,}")
    print(f"Percentage pruned : {100 * zero_params / total_params:.2f}%")
    return total_params, zero_params

total_params, zero_params = count_zeroed_and_total_weights(model)


print(f"k percent: {k_percent}")
print(f"Final Validation Accuracy: {val_accuracy}")
#score = (accuracy + num zero weights / total parameters) / 2

score = (val_accuracy/100 + zero_params / total_params ) / 2
print("Score must be .36 or higher")
print(f"Final score for assignment: {score}")

Total weights     : 592,224
Zeroed weights    : 29,826
Percentage pruned : 5.04%
k percent: 0.2
Final Validation Accuracy: 58.772277227722775
Score must be .36 or higher
Final score for assignment: 0.31904273643850045


RESULTS:

with masked weights and average activations and pruning schedule: .20 > 0.20 >

with masked weights and average activations and pruning schedule: .10 > .10 > .10 > .10 > 0.10 > 0.2 (2 fine tuning)
subset 200, Val acc = 60.23, score = 0.329, % pruned = 5.58%

with masked weights and average activations:

subset 200, k = 0.90: Val acc = 19.25, score = 0.14526, % pruned = 9.81%
subset 200, k = 0.80: Val acc = 29.35, score = 0.19058, % pruned = 8.77%
subset 200, k = 0.70: Val acc = 37.23, score = 0.22408, % pruned = 7.59%
subset 200, k = 0.60: Val acc = 43.64, score = 0.25099, % pruned = 6.55%
subset 200, k = 0.50: Val acc = 53.98, score = 0.29750, % pruned = 5.52%
subset 200, k = 0.40: Val acc = 60.12, score = 0.32202, % pruned = 4.29%
subset 200, k = 0.30: Val acc = 63.12, score = 0.33190, % pruned = 3.25%
subset 200, k = 0.20: Val acc = 65.12, score = 0.33727, % pruned = 2.07%
subset 200, k = 0.10: Val acc = 67.45, score = 0.34240, % pruned = 1.03%
subset 200, k = 0.05: Val acc = 68.95, score = 0.34721, % pruned = 0.49%
subset 200, k = 0.00: Val acc = 69.66, score = 0.34832, % pruned = 0.00%

With masked weights and max activation:

subset 200, k = 0.20: Val acc = 67.36, score = 0.34718, % pruned = 2.07%
subset 200, k = 0.10: Val acc = 68.20, score = 0.34616, % pruned = 1.03%


Without masked weights:

subset 200, k = 0.45: Val acc = 58.38, score = 0.31803
subset 200, k = 0.35: Val acc = 59.19, score = 0.32575
subset 200, k = 0.30: Val acc = 61.90, score = 0.32575
subset 200, k = 0.25: Val acc = 62.81, score = 0.32785
subset 200, k = 0.22: Val acc = 63.37, score = 0.3289
subset 200, k = 0.20: Val acc = 64.20, score = 0.3313
subset 200, k = 0.19: Val acc = 63.88, score = 0.32975
subset 200, k = 0.10: Val acc = 67.72, score = 0.34279
subset 200, k = 0.00: Val acc = 69.54, score = 0.34772


Ideas to Try:

Layer-wise Adaptive k_percent Pruning

Early Pruning + Intermediate Pruning + Late Prunin